In [1]:
import torch
import lightning as pl
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
import albumentations as A 
from albumentations.pytorch import ToTensorV2
import os 
import numpy as np
from PIL import Image
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import LearningRateMonitor
torch.set_float32_matmul_precision('high')

/home/linux/miniconda3/envs/P_3_11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ResidualLayer(torch.nn.Module):
    def __init__(self,channels_in,channels_out,pooling,kernel_size,activation_func):
        super().__init__()
        self.first_layer = torch.nn.Conv2d(channels_in,channels_out,kernel_size=kernel_size,padding=kernel_size//2,bias=True)
        self.out_channels=channels_out
        self.pooling = pooling
        self.b = torch.nn.BatchNorm2d(self.out_channels)
        self.activation_func = activation_func
        self.downsample = torch.nn.Identity() if channels_in==channels_out else torch.nn.Sequential(torch.nn.Conv2d(channels_in,channels_out,1,bias=False),torch.nn.BatchNorm2d(self.out_channels))
    def forward(self,X):
        temp_res = self.first_layer(X) 
        temp_res = self.b(temp_res)
        temp_res = self.activation_func(self.downsample(X) +temp_res)
       
       
        return self.pooling(temp_res)
        

In [ ]:


class meUnet(pl.LightningModule):
    def __init__(
        self, 
        channel_in, 
        channels_out,
        kernel_size,   
        activation_func=torch.nn.ReLU(), 
        pooling=torch.nn.MaxPool2d(2), 
        kernel_chain=[32, 64, 128], 
        optimizer='AdamW',
        lr=1e-3, 
        scheduler='none'
    ):
        super().__init__()
  
        
        self.channels_in = channel_in
        self.channels_out = channels_out
        self.kernel_chain = kernel_chain
        self.optimizer_name = optimizer
        self.lr = lr
        self.scheduler_name = scheduler

        self.encoder = torch.nn.ModuleList() 
        self.encoder.append(ResidualLayer(channel_in, kernel_chain[0], pooling, kernel_size, activation_func))
        for i in range(len(kernel_chain)-1):
            self.encoder.append(ResidualLayer(kernel_chain[i], kernel_chain[i+1], pooling, kernel_size, activation_func))
            
      
        self.decoder = torch.nn.ModuleList()
        for i in range(len(kernel_chain)-1, 0, -1):
            self.decoder.append(torch.nn.ConvTranspose2d(kernel_chain[i], kernel_chain[i-1], kernel_size=2, stride=2))
            self.decoder.append(ResidualLayer(kernel_chain[i-1]*2, kernel_chain[i-1], pooling=torch.nn.Identity(), kernel_size=kernel_size, activation_func=activation_func))

    
        self.fin_uno = torch.nn.ConvTranspose2d(kernel_chain[0], kernel_chain[0], kernel_size=2, stride=2)
        self.fin_dos = ResidualLayer(kernel_chain[0] + channel_in, kernel_chain[0], torch.nn.Identity(), kernel_size, activation_func)
        self.fin_tres = torch.nn.Conv2d(kernel_chain[0], channels_out, kernel_size=1)

    def forward(self, x):
       
        initial_x = x  
        skip_cons = []
        
       
        current = x
        for layer in self.encoder:
            current = layer(current)
            skip_cons.append(current) 
        
       
        skips = skip_cons[::-1] 
        x = skips[0] 
        
        skip_idx = 1
        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x) 
            x = torch.cat([skips[skip_idx], x], dim=1) 
            x = self.decoder[i+1](x) 
            skip_idx += 1

        
        x = self.fin_uno(x)
        x = torch.cat([initial_x, x], dim=1) 
        x = self.fin_dos(x)
        x = self.fin_tres(x)
        
        return x

    def configure_optimizers(self):
        
        match self.optimizer_name.lower():
            case 'adamw':
                o = torch.optim.AdamW(self.parameters(), lr=self.lr)
            case 'sgd_with_momentum':
                o = torch.optim.SGD(self.parameters(), lr=self.lr, momentum=0.9)
            case 'adam':
                o = torch.optim.Adam(self.parameters(), lr=self.lr)
            case 'nadam':
                o = torch.optim.NAdam(self.parameters(), lr=self.lr)
            case _:
                o = torch.optim.ASGD(self.parameters(), lr=self.lr)
                
       
        match self.scheduler_name.lower():
            case 'none':
                return o
            case 'step':
                return [o], [torch.optim.lr_scheduler.StepLR(o, step_size=10, gamma=0.1)]
            case 'cosine':
                return [o], [torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=10)]
            case 'reduce_on_plateau':
                s = torch.optim.lr_scheduler.ReduceLROnPlateau(o, mode='min')
                return {
                    "optimizer": o,
                    "lr_scheduler": {
                        "scheduler": s,
                        "monitor": "val_loss",
                    },
                }
            case _:
                return o

    def dice_score(self, probs: torch.Tensor, mask: torch.Tensor, epsilon=1e-6):
        props = probs.flatten(start_dim=1)
        mask = mask.flatten(start_dim=1)
        union_part = 2 * (props * mask).sum(dim=1)
        bottom_part = epsilon + (props.sum(dim=1) + mask.sum(dim=1))
        dice = (union_part + epsilon) / bottom_part
        return dice.mean()

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()
        logits = self(x)
        
        bcl = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
        probs = torch.sigmoid(logits)
        dice_loss = 1 - self.dice_score(probs=probs, mask=y)
        total_loss = bcl + dice_loss
        
      
        self.log("train_loss", total_loss, prog_bar=True, on_step=False, on_epoch=True)
        return total_loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()
        logits = self(x)
        probs = torch.sigmoid(logits)
        
       
        bcl_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
        dice = self.dice_score(probs=probs, mask=y)
        
        self.log("test_loss", bcl_loss + (1 - dice), prog_bar=True)
        self.log("test_dice", dice, prog_bar=True)
        
        return {"test_dice": dice}

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.float()
        logits = self(x)
        
        bcl_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
        probs = torch.sigmoid(logits)
        
    
        dice = self.dice_score(probs=probs, mask=y)
        dice_loss = 1 - dice
        total_loss = bcl_loss + dice_loss
        
       
        self.log("val_loss", total_loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("val_dice", dice, prog_bar=True, on_step=True, on_epoch=True)
        
        return total_loss

In [4]:


class meDataset(Dataset):
    def __init__(self, data_dir, transforms=None):
        self.img_dir = os.path.join(data_dir, "images")
        self.labels_dir = os.path.join(data_dir, "labels")
        self.transforms = transforms
        
        self.img_files = sorted([f for f in os.listdir(self.img_dir) if not f.startswith('.')])
        self.lab_files = sorted([f for f in os.listdir(self.labels_dir) if not f.startswith('.')])
        
        self.length = min(len(self.img_files), len(self.lab_files))

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        
        img_path = os.path.join(self.img_dir, self.img_files[index])
        mask_path = os.path.join(self.labels_dir, self.lab_files[index])

        img = np.array(Image.open(img_path).convert("L"), dtype=np.float32) / 255.0
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        
  
        mask[mask > 0] = 1.0

        if self.transforms:
            augments = self.transforms(image=img, mask=mask)
            img = augments['image']
            mask = augments['mask']

        if not isinstance(img, torch.Tensor):
            img = torch.from_numpy(img)
        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask)

        if img.dim() == 2:
            img = img.unsqueeze(0)
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
            
        return img, mask

In [ ]:
class meDataModule(pl.LightningDataModule):
    def __init__(self, train_path, val_path, test_path, train_transforms, validation_transforms, batch_size=8):
        super().__init__()
        self.train_path = train_path
        self.val_path = val_path
        self.test_path = test_path
        self.batch_size = batch_size
        self.train_transforms = train_transforms
        self.validation_transforms = validation_transforms
        
       
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            self.train_ds = meDataset(self.train_path, transforms=self.train_transforms)
            self.val_ds = meDataset(self.val_path, transforms=self.validation_transforms)

        if stage == "test" or stage is None:
           
            self.test_ds = meDataset(self.test_path, transforms=self.validation_transforms)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def test_dataloader(self):
     
        if self.test_ds is None:
            self.setup(stage="test")
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=4)

In [6]:
train_transform = A.Compose([
    A.RandomResizedCrop(
        size=(128,128),
        scale=(0.3, 1.0),
        p=0.7
    ),
    A.Resize(128,128),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(0.25),
    A.Affine(
        translate_percent={"x": (-0.0625, 0.0625), "y": (-0.0625, 0.0625)},
        scale=(0.9, 1.1),
        rotate=(-45, 45),
        p=0.3
    ),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(std_range=(0.001, 0.005), p=0.3),
    ToTensorV2()
])
validation_transforms = A.Compose([
    A.Resize(128,128),
    ToTensorV2()
])

In [7]:
train_data = meDataset("dataset/train",transforms=train_transform)
test_ds = meDataset("dataset/test",transforms=validation_transforms)
val_ds = meDataset("dataset/val",transforms=validation_transforms)

In [8]:
"""
meModel=meUnet(channel_in=1,channels_out=1,kernel_size=3,kernel_chain=[32,64,128],optimizer='other',lr=1e-4,scheduler='reduce_on_plateau')
meData = meDataModule(
    train_path="dataset/train",
    val_path="dataset/val",
    test_path="dataset/test",
    train_transforms=train_transform,
    validation_transforms=validation_transforms,
    batch_size=8)
"""

'\nmeModel=meUnet(channel_in=1,channels_out=1,kernel_size=3,kernel_chain=[32,64,128],optimizer=\'other\',lr=1e-4,scheduler=\'reduce_on_plateau\')\nmeData = meDataModule(\n    train_path="dataset/train",\n    val_path="dataset/val",\n    test_path="dataset/test",\n    train_transforms=train_transform,\n    validation_transforms=validation_transforms,\n    batch_size=8)\n'

In [9]:
import optuna 
from optuna import Trial
import optuna.visualization as vis

In [ ]:
import time
from lightning.pytorch.callbacks import Callback

class CoolingBreakCallback(Callback):
    def __init__(self, run_duration_min=5, break_duration_min=1):
        super().__init__()
        self.run_duration = run_duration_min * 60
        self.break_duration = break_duration_min * 60
        self.last_break_time = time.time()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
       
        if time.time() - self.last_break_time > self.run_duration:
            print(f"\n[GPU Cooling] Taking a {self.break_duration/60} min break...")
            time.sleep(self.break_duration)
            self.last_break_time = time.time()  #
            print("[GPU Cooling] Resuming training.")




In [ ]:

def objective(trial: Trial):
    activation_funcs = {
        0: torch.nn.Tanh(),
        1: torch.nn.Sigmoid(),
        2: torch.nn.ReLU(),
        
    }
    poolings = {
        0: torch.nn.MaxPool2d(2),
        1: torch.nn.AvgPool2d(2),
        2: torch.nn.FractionalMaxPool2d(2,output_ratio=0.5)
    
    }
    cooling_callback = CoolingBreakCallback(run_duration_min=5, break_duration_min=1.3)
    optim_name = trial.suggest_categorical('optimizer', ['adam', 'adamw', 'sgd_with_momentum', 'nadam'])
    scheduler_name = trial.suggest_categorical('scheduler', ['none', 'reduce_on_plateau', 'cosine', 'step'])
    lr = trial.suggest_float("lr", 1e-6, 1e-2, log=True)
    
    act_idx = trial.suggest_int('activation_func', 0, 2)
    selected_activation = activation_funcs[act_idx]
    pooling_idx = trial.suggest_int('pooling', 0, 2)
    selected_pooling = poolings[pooling_idx]

    num_of_layers = trial.suggest_int('num_of_layers', 1, 7, step=1)
    start_kernels = trial.suggest_int('start_kernel', 16, 64)
    kernel_growth = trial.suggest_int('step', 16, 64)
    
    stop_val = start_kernels + (num_of_layers * kernel_growth)
    kernel_chain = np.arange(start_kernels, stop_val, kernel_growth)

    mlf_logger = MLFlowLogger(
        experiment_name="meUnet-testsv2",
        tracking_uri="http://localhost:5000",
        run_name=f"run numero -> {trial.number}"
    )

    model = meUnet(
        channel_in=1,
        channels_out=1,
        kernel_size=3,
        activation_func=selected_activation,
        kernel_chain=kernel_chain,
        pooling=selected_pooling,
        optimizer=optim_name,
        lr=lr,
        scheduler=scheduler_name
    )

    meData = meDataModule(
        train_path="dataset/train",
        val_path="dataset/val",
        test_path="dataset/test",
        train_transforms=train_transform,
        validation_transforms=validation_transforms,
        batch_size=8
    )

    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    pruning_callback = optuna.integration.PyTorchLightningPruningCallback(trial, monitor="val_loss")

    trainer = pl.Trainer(
        max_epochs=80,
        accelerator="auto",
        devices=1,
        logger=mlf_logger,
        log_every_n_steps=10,
        callbacks=[lr_monitor, pruning_callback,cooling_callback],
        enable_checkpointing=False,
        precision=16 if torch.cuda.is_available() else 32
    )

    mlf_logger.log_hyperparams({
        "num_layers": num_of_layers,
        "kernel_chain": str(kernel_chain),
        "activation": type(selected_activation).__name__
        # me forgot about the pooling :C
    })

    try:
        trainer.fit(model, datamodule=meData)
    except Exception as e:
        return float("inf")

    return trainer.callback_metrics["val_loss"].item()

In [14]:
study = optuna.create_study(study_name='testing_unet',direction='minimize',load_if_exists=True)
study.optimize(objective, n_trials=80)

[I 2026-03-29 13:06:09,967] A new study created in memory with name: testing_unet
/home/linux/miniconda3/envs/P_3_11/lib/python3.11/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type            | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder  | ModuleList      | 1.0 M  | train | 0    
1 | decoder  | ModuleList      | 2.0 M  | train | 0    
2 | fin_uno  | ConvTranspose2d | 3.2 K  | train | 0    
3 | fin_

Epoch 79: 100%|██████████| 27/27 [00:03<00:00,  8.60it/s, v_num=ab91, val_loss_step=0.491, val_dice_step=0.682, val_loss_epoch=0.297, val_dice_epoch=0.776, train_loss=0.402]

`Trainer.fit` stopped: `max_epochs=80` reached.


Epoch 79: 100%|██████████| 27/27 [00:03<00:00,  8.55it/s, v_num=ab91, val_loss_step=0.491, val_dice_step=0.682, val_loss_epoch=0.297, val_dice_epoch=0.776, train_loss=0.402]
🏃 View run run numero -> 0 at: http://localhost:5000/#/experiments/2/runs/cac39233240f4f28a97cfe5455adab91
🧪 View experiment at: http://localhost:5000/#/experiments/2


[I 2026-03-29 13:10:28,745] Trial 0 finished with value: 0.29657313227653503 and parameters: {'optimizer': 'sgd_with_momentum', 'scheduler': 'reduce_on_plateau', 'lr': 0.00021076057086073644, 'activation_func': 2, 'pooling': 0, 'num_of_layers': 6, 'start_kernel': 28, 'step': 42}. Best is trial 0 with value: 0.29657313227653503.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type            | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder  | ModuleList      | 114 K  | train | 0    
1 | decoder  | ModuleList      | 216 K  | train | 0    
2 | fin_uno  | ConvTranspose2d | 11.7 K | t

...

[I 2026-03-29 18:21:27,222] Trial 77 finished with value: inf and parameters: {'optimizer': 'adam', 'scheduler': 'cosine', 'lr': 0.0008779183623136154, 'activation_func': 2, 'pooling': 2, 'num_of_layers': 5, 'start_kernel': 55, 'step': 23}. Best is trial 36 with value: 0.0787193700671196.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type            | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder  | ModuleList      | 547 K  | train | 0    
1 | decoder  | ModuleList      | 1.1 M  | train | 0    
2 | fin_uno  | ConvTranspose2d | 9.3 K  | train | 0    
3 | fin_dos  | ResidualLaye

Epoch 0: 100%|██████████| 27/27 [00:08<00:00,  3.25it/s, v_num=32fd]       🏃 View run run numero -> 78 at: http://localhost:5000/#/experiments/2/runs/d13ed041d70044b3b7f4518ddbaa32fd
🧪 View experiment at: http://localhost:5000/#/experiments/2


[I 2026-03-29 18:21:47,982] Trial 78 finished with value: inf and parameters: {'optimizer': 'adam', 'scheduler': 'reduce_on_plateau', 'lr': 0.001533923121685783, 'activation_func': 2, 'pooling': 2, 'num_of_layers': 6, 'start_kernel': 48, 'step': 21}. Best is trial 36 with value: 0.0787193700671196.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type            | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder  | ModuleList      | 471 K  | train | 0    
1 | decoder  | ModuleList      | 934 K  | train | 0    
2 | fin_uno  | ConvTranspose2d | 11.3 K | train | 0    
3 | fin_dos  | Re

Epoch 3: 100%|██████████| 27/27 [00:04<00:00,  6.54it/s, v_num=861b, val_loss_step=0.854, val_dice_step=0.385, val_loss_epoch=0.700, val_dice_epoch=0.464, train_loss=0.847]🏃 View run run numero -> 79 at: http://localhost:5000/#/experiments/2/runs/a959a77c06f8455f8168ff6585dd861b
🧪 View experiment at: http://localhost:5000/#/experiments/2


[I 2026-03-29 18:22:24,398] Trial 79 finished with value: inf and parameters: {'optimizer': 'adam', 'scheduler': 'reduce_on_plateau', 'lr': 0.0002653370773796106, 'activation_func': 2, 'pooling': 2, 'num_of_layers': 5, 'start_kernel': 53, 'step': 26}. Best is trial 36 with value: 0.0787193700671196.


In [15]:
fig_importance = vis.plot_param_importances(study)
fig_importance.show()

[W 2026-03-29 18:22:24,415] Trial 5 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,416] Trial 6 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,417] Trial 8 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,417] Trial 9 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,418] Trial 10 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,418] Trial 11 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,419] Trial 14 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,419] Trial 15 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,420] Trial 17 is omitted in visualization because its objective value is inf or nan.
[W 2026-03-29 18:22:24,421] Tria

In [16]:
fig_history = vis.plot_optimization_history(study)
fig_history.show()

In [24]:
study.best_params

{'optimizer': 'adamw',
 'scheduler': 'reduce_on_plateau',
 'lr': 0.00021763716310291566,
 'activation_func': 2,
 'pooling': 2,
 'num_of_layers': 5,
 'start_kernel': 56,
 'step': 40}

In [32]:
kernels = np.arange(56,56+40*5,40)

In [33]:
theBestModel = meUnet(1,1,3,optimizer='adamw',lr=0.00021763716310291566,activation_func=torch.nn.ReLU(),pooling=torch.nn.FractionalMaxPool2d(2,output_ratio=0.5),kernel_chain=kernels)

In [34]:
trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices=1
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [35]:
meData = meDataModule(
        train_path="dataset/train",
        val_path="dataset/val",
        test_path="dataset/test",
        train_transforms=train_transform,
        validation_transforms=validation_transforms,
        batch_size=8
    )

In [36]:
trainer.fit(theBestModel ,datamodule=meData)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type            | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder  | ModuleList      | 807 K  | train | 0    
1 | decoder  | ModuleList      | 1.6 M  | train | 0    
2 | fin_uno  | ConvTranspose2d | 12.6 K | train | 0    
3 | fin_dos  | ResidualLayer   | 32.2 K | train | 0    
4 | fin_tres | Conv2d          | 57     | train | 0    
-------------------------------------------------------------
2.4 M     Trainable params
0         Non-trainable params
2.4 M     Total params
9.654     Total estimated model params size (MB)
75        Modules in train mode
0         Modules in eval mode
0         Total Flops


/home/linux/miniconda3/envs/P_3_11/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (27) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 99: 100%|██████████| 27/27 [00:06<00:00,  3.98it/s, v_num=33, val_loss_step=0.0551, val_dice_step=0.962, val_loss_epoch=0.0753, val_dice_epoch=0.945, train_loss=0.105] 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 27/27 [00:06<00:00,  3.92it/s, v_num=33, val_loss_step=0.0551, val_dice_step=0.962, val_loss_epoch=0.0753, val_dice_epoch=0.945, train_loss=0.105]


In [37]:
trainer.test(theBestModel , datamodule=meData)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: 100%|██████████| 6/6 [00:03<00:00,  1.83it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_dice           0.9152007102966309
        test_loss           0.11144564300775528
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.11144564300775528, 'test_dice': 0.9152007102966309}]

In [43]:
torch.serialization.add_safe_globals([
    'numpy._core.multiarray._reconstruct',
    'numpy.ndarray',
    'numpy.dtype'
])

In [44]:
trainer.save_checkpoint("meModel.ckpt")

`weights_only` was not set, defaulting to `False`.


In [46]:
model_p = meUnet.load_from_checkpoint("meModel.ckpt",weights_only=False)